In [1]:
import h5py
import numpy as np
import joblib
import torch

In [2]:
print(torch.cuda.is_available())

True


In [3]:
DS_ROOT = '/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_0.05_run_jump_forward_backward'

In [4]:
meta_data = joblib.load(f'{DS_ROOT}/phc_act_amass_train_upright_metadata.pkl')
failed_keys = joblib.load(f'{DS_ROOT}/failed.pkl')

In [5]:
print(meta_data.keys())

motion_lengths = np.concatenate( [ml for ml in meta_data['motion_lengths']])
keys_names = np.concatenate( [kn for kn in meta_data['key_names']])

print(len(motion_lengths))
print(sum(motion_lengths))
print(len(keys_names))
print(keys_names)
num_motions = len(keys_names)
motion_starts = np.cumsum(motion_lengths)
motion_starts = np.insert(motion_starts, 0, 0)
motion_starts = motion_starts[:-1]
print(motion_starts[:5])

dict_keys(['actions', 'key_names', 'motion_lengths', 'reset', 'running_mean', 'config'])
800
239200
800
['0-KIT_1297_wipe04_poses_sample_0'
 '0-KIT_575_MarcusS_AdrianM09_poses_sample_0'
 '0-KIT_513_parkour03_poses_sample_0'
 '0-KIT_425_pour_and_mixing_different_speeds_01_poses_sample_0'
 '0-CMU_40_40_04_poses_sample_0' '0-CMU_142_142_16_poses_sample_0'
 '0-CMU_142_142_11_poses_sample_1' '0-CMU_132_132_50_poses_sample_0'
 '0-CMU_132_132_39_poses_sample_0' '0-CMU_132_132_44_poses_sample_0'
 '0-KIT_513_seesaw_up07_poses_sample_0'
 '0-KIT_513_step_stones08_poses_sample_0'
 '0-KIT_674_wash_back02_poses_sample_1' '0-CMU_143_143_39_poses_sample_0'
 '0-CMU_141_141_31_poses_sample_0' '0-CMU_40_40_05_poses_sample_0'
 '0-CMU_40_40_02_poses_sample_0' '0-CMU_61_61_15_poses_sample_0'
 '0-CMU_41_41_02_poses_sample_1' '0-CMU_17_17_07_poses_sample_1'
 '0-KIT_1297_wipe04_poses_sample_0'
 '0-KIT_575_MarcusS_AdrianM09_poses_sample_0'
 '0-KIT_513_parkour03_poses_sample_0'
 '0-KIT_425_pour_and_mixing_differ

In [6]:
print(len(failed_keys))
print(failed_keys)

0
[]


In [9]:
dataset_path = f'{DS_ROOT}/phc_act_amass_train_upright.h5'
with h5py.File(dataset_path, 'r') as hdf5_file:
    # Get the total size of the dataset
    for key in hdf5_file.keys():
        print(f"- {key}")
    dataset_size = len(hdf5_file['clean_action']) 

    assert dataset_size == sum(motion_lengths)
    # Load the `reset` boolean array
    reset = hdf5_file['reset'][:]
    action  =  hdf5_file['clean_action'][:]
    obs  =  hdf5_file['pdp_obs'][:]



- actions
- clean_action
- pdp_obs
- pdp_ref
- reset


In [11]:

exclude_ids = []
exclude_indicies = []

obs_dim = obs.shape[-1]
action_dim = action.shape[-1]

count_obs = 0
mean_obs = np.zeros(obs_dim, dtype=obs.dtype)
M2_obs = np.zeros(obs_dim, dtype=obs.dtype)
# Initialize min and max as before
obs_min = np.full(obs_dim, np.inf, dtype=obs.dtype)
obs_max = np.full(obs_dim, -np.inf, dtype=obs.dtype)


count_action = 0
mean_action = np.zeros(action_dim, dtype=obs.dtype)
M2_action = np.zeros(action_dim, dtype=obs.dtype)
# Initialize min and max as before
action_min = np.full(action_dim, np.inf, dtype=obs.dtype)
action_max = np.full(action_dim, -np.inf, dtype=obs.dtype)

for m_i in range(num_motions):

    ml = motion_lengths[m_i]
    start_idx, end_idx = motion_starts[m_i], motion_starts[m_i]+ml


    if keys_names[m_i] in failed_keys:
        exclude_ids.append(m_i)
        exclude_indicies.extend(range(start_idx, end_idx))
        continue
    

    m_r = reset[start_idx:end_idx]
    m_a = action[start_idx:end_idx]
    m_o = obs[start_idx:end_idx]

    has_no_reset = (np.sum(m_r) == 0)
    if has_no_reset:
        exclude_ids.append(m_i)
        exclude_indicies.extend(range(start_idx, end_idx))
        continue

    is_last_a_reset = (m_r[-1] == 1)
    assert is_last_a_reset, 'Last index is not Reset'

    has_only_one_reset = (np.sum(m_r) == 1)
    assert has_only_one_reset, 'More the one Reset in data'

    #OBS -------------- Update min/max
    obs_min = np.minimum(obs_min, m_o.min(axis=0))
    obs_max = np.maximum(obs_max, m_o.max(axis=0))

    # Update mean and M2 using Welford's algorithm -------------
    chunk_count = m_o.shape[0]
    
    delta = m_o - mean_obs
    new_mean_obs = mean_obs + np.sum(delta, axis=0) / (count_obs + chunk_count)
    
    delta2 = m_o - new_mean_obs
    M2_obs += np.sum(delta * delta2, axis=0)
    
    mean_obs = new_mean_obs
    count_obs += chunk_count


    #OBS -------------- Update min/max ----------------
    action_min = np.minimum(action_min, m_a.min(axis=0))
    action_max = np.maximum(action_max, m_a.max(axis=0))

    # Update mean and M2 using Welford's algorithm
    chunk_count = m_o.shape[0]
    
    delta = m_a - mean_action
    new_mean_action = mean_action + np.sum(delta, axis=0) / (count_action + chunk_count)
    
    delta2 = m_a - new_mean_action
    M2_action += np.sum(delta * delta2, axis=0)
    
    mean_action = new_mean_action
    count_action += chunk_count



all_indices = np.arange(len(action))
indices_to_keep = np.setdiff1d(all_indices, exclude_indicies)

std_obs = np.sqrt(M2_obs / count_obs)
std_action = np.sqrt(M2_action / count_action)


In [12]:
import sys
import os


module_path = os.path.abspath('/home/mcarroll/Documents/cd-2/VideoMimic/PDP')

# Insert the path at the beginning of the list
sys.path.insert(0, module_path)
from pdp.utils.normalizer import LinearNormalizer

In [13]:
data = {
    'obs': {
        'min':obs_min,
        'max':obs_max,
        'mean':mean_obs,
        'std':std_obs,
    },
    'action': {
        'min':action_min,
        'max':action_max,
        'mean':mean_action,
        'std':std_action,
    },
}

normalizer = LinearNormalizer()
normalizer.fit_implicit(data=data, mode='limits')


{'min': array([-7.50166103e-02,  1.26491375e-02, -1.12548843e-01, -2.11369514e-01,
       -3.88090238e-02, -4.88148689e-01, -4.96876627e-01, -2.20558539e-01,
       -8.73527527e-01, -5.41130424e-01, -1.65488213e-01, -9.90362346e-01,
       -7.46154487e-02, -1.04584590e-01, -1.10430509e-01, -1.66457623e-01,
       -3.66015226e-01, -4.88246024e-01, -5.00796616e-01, -6.23425543e-01,
       -8.74415219e-01, -6.14395440e-01, -7.40939200e-01, -9.99187768e-01,
       -7.26596490e-02, -5.17221242e-02,  7.83182532e-02, -9.90381539e-02,
       -8.82136002e-02,  7.32369423e-02, -7.12552518e-02, -1.05045065e-01,
        4.26999964e-02, -1.23632483e-01, -2.52041727e-01,  5.99327758e-02,
       -8.59712586e-02, -2.92418480e-01,  2.20531248e-03, -1.23858534e-01,
       -1.29802048e-01,  7.58001208e-02, -1.45346373e-01, -4.84263673e-02,
        1.63499676e-02, -3.16911370e-01, -1.83772668e-02, -2.25682139e-01,
       -4.48467791e-01, -2.36637712e-01, -4.43022221e-01, -5.08178473e-01,
       -2.8351208

In [14]:
normalizer_state = normalizer.state_dict()

# Save the state dictionary to a file
torch.save(normalizer_state, f'{DS_ROOT}/normalizer_params.pt')

In [15]:
meta_data['exclude_ids'] = exclude_ids
joblib.dump(meta_data, f'{DS_ROOT}/phc_act_amass_train_upright_metadata.pkl')

['/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_0.05_run_jump_forward_backward/phc_act_amass_train_upright_metadata.pkl']

In [16]:
with h5py.File("/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_0.05_run_jump_forward_backward/phc_act_amass_train_upright.h5", "r") as f_in, \
    h5py.File("/home/mcarroll/Documents/cd-2/VideoMimic/PDP/data/phc_0.05_run_jump_forward_backward/phc_act_amass_train_upright_v2.h5", "w") as f_out:
    for k, dset in f_in.items():
        if k in ["pdp_obs", "clean_action", "pdp_ref"]:
            shape = dset.shape
            dtype = dset.dtype
            chunk_len = 64  # or your horizon
            chunk_shape = (min(chunk_len, shape[0]),) + shape[1:]

            new_dset = f_out.create_dataset(
                k,
                shape=shape,
                dtype=dtype,
                chunks=chunk_shape,
                compression="lzf",
                shuffle=True
            )

            # Read everything into numpy first, then write
            new_dset[:] = dset[:]